# 04. Baseline Model Training, Calibration & TreeSHAP Explainability
## Water Quality Prediction via Gradient Boosting & Deep Neural Networks (Phase 1)

### Objectives:
- Benchmark tree-based AML baselines (**Random Forest** and **XGBoost**) against a baseline **Dummy** predictor.
- Evaluate both target paradigms:
  1. **Continuous Target**: CCME Values Index (evaluated via MAE, RMSE, $R^2$, MAPE).
  2. **Discrete Target**: CCME Water Quality Class (evaluated via Macro F1, Balanced Accuracy, Brier Score).
- Quantify probabilistic calibration via **Expected Calibration Error (ECE)** and Reliability Diagrams.
- Explain feature contributions and sensor interactions via coalitional game-theoretic **TreeSHAP**.
- Analyze the theoretical inductive bias gap motivating Deep Neural Networks (MLP) in Phase 2.

In [ ]:
import sys
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.models.evaluate import compute_regression_metrics, compute_classification_metrics, plot_calibration_curves
from src.models.explainability import explain_model_with_shap

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
print("Model evaluation and explainability environment initialized!")

### 1. Load Evaluation Results & Metrics

In [ ]:
eval_path = ROOT_DIR / "reports" / "figures" / "evaluation_summary.json"
if eval_path.exists():
    with open(eval_path, 'r') as f:
        eval_summary = json.load(f)
    
    print("--- CONTINUOUS WQI REGRESSION BENCHMARK ---")
    df_reg = pd.DataFrame(eval_summary['regression']).T
    display(df_reg)
    
    print("\n--- DISCRETE WQI CLASSIFICATION BENCHMARK ---")
    df_cls = pd.DataFrame(eval_summary['classification']).T
    display(df_cls)
else:
    print("Run 'python -m src.main_phase1' to generate full trained model evaluation summaries.")

### 2. Inspect Probabilistic Calibration Curves (Reliability Diagram)
In water resource management, decision thresholds depend on calibrated probability outputs. Expected Calibration Error (ECE) measures deviation from ideal diagonal confidence alignment.

In [ ]:
calib_fig_path = ROOT_DIR / "reports" / "figures" / "model_calibration_curves.png"
if calib_fig_path.exists():
    from IPython.display import Image
    display(Image(filename=str(calib_fig_path)))
else:
    print("Calibration curve figure not yet generated.")

### 3. TreeSHAP Global Feature Importance (Mean |SHAP|)
TreeSHAP calculates exact Shapley attributions without the cardinality bias of Gini impurity.

In [ ]:
bar_fig_path = ROOT_DIR / "reports" / "figures" / "shap_bar_xgboost_regression.png"
if bar_fig_path.exists():
    from IPython.display import Image
    display(Image(filename=str(bar_fig_path)))
else:
    print("SHAP bar plot figure not yet generated.")

### 4. TreeSHAP Summary Beeswarm Plot
Shows the directional effect of each sensor: red denotes higher sensor readings, blue denotes lower readings. Positive SHAP values increase predicted WQI, negative SHAP values decrease it.

In [ ]:
beeswarm_fig_path = ROOT_DIR / "reports" / "figures" / "shap_beeswarm_xgboost_regression.png"
if beeswarm_fig_path.exists():
    from IPython.display import Image
    display(Image(filename=str(beeswarm_fig_path)))
else:
    print("SHAP beeswarm figure not yet generated.")

### 5. Theoretical Synthesis: The "Gap" Leading to Phase 2
1. **Tree Strengths:** Invariance to monotonic scalings, rapid feature partition thresholds, robust handling of unnormalized tabular inputs.
2. **Tree Limitations:** Decision trees partition feature space strictly via axis-aligned orthogonal cuts ($x_j \le \theta$). Continuous chemical interaction kinetics (such as oxygen solubility curves and multi-nutrient buffering) require an exponential number of split staircases to approximate smooth diagonal manifolds.
3. **Phase 2 Deep Neural Network (MLP) Outlook:**
   - Deploy a small regularized MLP with LayerNorm/BatchNorm and smooth activations (GELU/SiLU).
   - Implement continuous regression head for WQI alongside a classification head for water quality classes.
   - Compare generalization performance on identical holdout splits to empirically evaluate the tree-to-neural gap.